In [10]:
import time

from sklearn.linear_model import Ridge, LinearRegression
from sklearn.tree import DecisionTreeRegressor

import optuna

from lineartree import (
    LinearTreeRegressor,
    SobolevForestRegressor,
)
import numpy as np
from sklearn.preprocessing import SplineTransformer
from sklearn.base import BaseEstimator, RegressorMixin
from sklearn.utils.validation import check_X_y, check_array, check_is_fitted
from sklearn.model_selection import KFold, cross_val_score

In [11]:
class TensorSplineRegressor(RegressorMixin, BaseEstimator):
    def __init__(self, n_knots=6, degree=3, knots="quantile", r_alpha=None):
        self.n_knots = n_knots
        self.degree = degree
        self.knots = knots
        self.r_alpha = r_alpha

    def _basis(self, X, alpha):
        blocks = [
            spline(X[:, j], nu=alpha[j])
            for j, spline in enumerate(self.spline_.bsplines_)
        ]

        basis = blocks[0]
        for block in blocks[1:]:
            basis = np.einsum(
                "ni,nj->nij", basis, block
            ).reshape(len(X), -1)

        return basis

    def fit(self, X, y):
        X, y = check_X_y(X, y)
        self.n_features_in_ = X.shape[1]

        self.spline_ = SplineTransformer(
            n_knots=self.n_knots,
            degree=self.degree,
            knots=self.knots,
            include_bias=True,
            extrapolation="continue",
        ).fit(X)

        basis = self._basis(X, (0,) * self.n_features_in_)

        if self.r_alpha is None:
            self.model_ = LinearRegression(
                fit_intercept=False
            ).fit(basis, y)
        else:
            self.model_ = Ridge(
                fit_intercept=False,
                alpha=self.r_alpha,
            ).fit(basis, y)

        return self

    def predict(self, X, alpha=None):
        check_is_fitted(self, "model_")
        X = check_array(X)

        alpha = (
            (0,) * self.n_features_in_
            if alpha is None
            else alpha
        )

        if len(alpha) != self.n_features_in_:
            raise ValueError("alpha must have one entry per feature")

        return self._basis(X, alpha) @ self.model_.coef_

In [12]:
def tune_forest(X, Y, k, T, tree_params, forest_params):
    space = {
        f"estimator__{p}": d for p, d in tree_params.items()
    } | forest_params

    model = SobolevForestRegressor(LinearTreeRegressor(
        Ridge(fit_intercept=False), derivative_weights=[0.0, 0.0]
    ))
    folds = KFold(k, shuffle=True, random_state=0)
    study = optuna.create_study()

    for _ in range(T):
        trial = study.ask(space)
        score = cross_val_score(
            model.set_params(**trial.params), X, Y,
            cv=folds, scoring="neg_mean_squared_error",
        ).mean()
        study.tell(trial, -score)
        # print('score:', -score)
        # print('param space forest:', trial.params)

    return model.set_params(**study.best_params).fit(X, Y), study

def tune_spline(X, Y, k, T, spline_params):
    model = TensorSplineRegressor()
    folds = KFold(k, shuffle=True, random_state=0)
    study = optuna.create_study()

    for _ in range(T):
        trial = study.ask(spline_params)
        score = cross_val_score(
            model.set_params(**trial.params), X, Y,
            cv=folds, scoring="neg_mean_squared_error",
        ).mean()
        study.tell(trial, -score)

    return model.set_params(**study.best_params).fit(X, Y), study

In [22]:
dgp = "smooth_regimes"  # or "regime_mixed"

from scipy.special import expit

def gate(x, c, k=25):
    s = expit(k * (x - c))
    ds = k * s * (1 - s)
    return s, ds, k * ds * (1 - 2 * s)

# Columns: level, u² coefficient, v² coefficient, uv coefficient
regime_coef = np.array([
    [ 0,   8,  -4,  6],
    [ 7, -10,   6, -8],
    [-6,   5,  12,  4],
    [10,  -6, -10,  8],
    [-8,  12,   4, -6],
    [ 4,  -8,  10, -4],
])

def f(x):
    u, v = x.T

    if dgp == "sine":
        su, sv = np.sin(2 * np.pi * u), np.sin(2 * np.pi * v)
        return su + sv + 0.5 * su * sv

    if dgp == "quartic":
        return (u**2 + v - 1)**2 + (u + v**2 - 1)**2

    if dgp == "exponential":
        return 2 * np.exp(2 * u * v - 2)

    if dgp == "regime_quadratic":
        r = np.searchsorted([0.35, 0.70], u) + 3 * (v > 0.55)
        m, a, b, c = regime_coef[r].T
        return (
            m + a * (u - 0.5)**2 + b * (v - 0.5)**2
            + c * (u - 0.5) * (v - 0.5)
        )

    if dgp == "regime_mixed":
        return np.select(
            [u < 0.4, v < 0.6],
            [
                6 + 3 * np.sin(2 * np.pi * u),
                -6 + 8 * (v - 0.3)**2 + 4 * u * v,
            ],
            default=(
                2 + 3 * np.exp(2 * (u - 0.7))
                + 20 * (v - 0.8)**3
            ),
        )

    if dgp == "smooth_regimes":
        s = gate(u, 0.4)[0]
        t = gate(v, 0.6)[0]
        return (
            u**2 + v**2
            + s * (3 - 3 * v**2)
            + t * (-5 + 4 * u**2)
        )

    if dgp == "gated_sine":
        s = gate(u, 0.6)[0]
        return u**2 + v + s * (3 + 2 * np.sin(4 * np.pi * v))

    raise ValueError(dgp)


def hessian_f(x):
    u, v = x.T
    H = np.empty((len(x), 2, 2))

    if dgp == "sine":
        w = 2 * np.pi
        su, sv = np.sin(w * u), np.sin(w * v)
        H[:, 0, 0] = -w**2 * su * (1 + 0.5 * sv)
        H[:, 1, 1] = -w**2 * sv * (1 + 0.5 * su)
        H[:, 0, 1] = H[:, 1, 0] = (
            0.5 * w**2 * np.cos(w * u) * np.cos(w * v)
        )

    elif dgp == "quartic":
        H[:, 0, 0] = 12 * u**2 + 4 * v - 2
        H[:, 1, 1] = 12 * v**2 + 4 * u - 2
        H[:, 0, 1] = H[:, 1, 0] = 4 * (u + v)

    elif dgp == "exponential":
        g = f(x)
        H[:, 0, 0] = 4 * v**2 * g
        H[:, 1, 1] = 4 * u**2 * g
        H[:, 0, 1] = H[:, 1, 0] = (2 + 4 * u * v) * g

    elif dgp == "regime_quadratic":
        r = np.searchsorted([0.35, 0.70], u) + 3 * (v > 0.55)
        _, a, b, c = regime_coef[r].T

        H[:, 0, 0] = 2 * a
        H[:, 1, 1] = 2 * b
        H[:, 0, 1] = H[:, 1, 0] = c

    elif dgp == "regime_mixed":
        regions = [u < 0.4, v < 0.6]

        H[:, 0, 0] = np.select(
            regions,
            [-12 * np.pi**2 * np.sin(2 * np.pi * u), 0],
            default=12 * np.exp(2 * (u - 0.7)),
        )
        H[:, 1, 1] = np.select(
            regions, [0, 16], default=120 * (v - 0.8)
        )
        H[:, 0, 1] = H[:, 1, 0] = np.select(
            regions, [0, 4], default=0
        )

    elif dgp == "smooth_regimes":
        s, ds, dds = gate(u, 0.4)
        t, dt, ddt = gate(v, 0.6)

        H[:, 0, 0] = 2 + dds * (3 - 3 * v**2) + 8 * t
        H[:, 1, 1] = 2 - 6 * s + ddt * (-5 + 4 * u**2)
        H[:, 0, 1] = H[:, 1, 0] = -6 * v * ds + 8 * u * dt

    elif dgp == "gated_sine":
        s, ds, dds = gate(u, 0.6)
        w = 4 * np.pi

        H[:, 0, 0] = 2 + dds * (3 + 2 * np.sin(w * v))
        H[:, 1, 1] = -2 * w**2 * s * np.sin(w * v)
        H[:, 0, 1] = H[:, 1, 0] = 2 * w * ds * np.cos(w * v)

    else:
        raise ValueError(dgp)

    return H


x_test_fixed = np.random.default_rng(2026).uniform(size=(50_000, 2))

def make_mc(seeds=range(100), n_train=1_000, sigma=0.2):
    mc = {}
    y_test = f(x_test_fixed)

    for seed in seeds:
        rng = np.random.default_rng(seed)
        x_train = rng.uniform(size=(n_train, 2))
        y_train = f(x_train) + rng.normal(scale=sigma, size=n_train)
        mc[seed] = (x_train, y_train, x_test_fixed, y_test)

    return mc

In [25]:
mse_20, mse_11, mse_02 = [[], []], [[], []], [[], []]
mae_20, mae_11, mae_02 = [[], []], [[], []], [[], []]

D = optuna.distributions

tree_params = {
    "max_depth": D.IntDistribution(3, 4),
    "min_samples_leaf": D.IntDistribution(20, 60),
    "max_features": D.CategoricalDistribution([2]),
    "base_estimator__alpha": D.FloatDistribution(1e-3, 0.1, log=True),
}

forest_params = {
    "max_samples": D.FloatDistribution(0.6, 0.9),
    "n_estimators": D.CategoricalDistribution([300]),
    "n_jobs": D.CategoricalDistribution([28]),
    "random_state": D.CategoricalDistribution([0]),
}

spline_params = {
    "n_knots": D.IntDistribution(3, 5),
    "knots": D.CategoricalDistribution(["uniform", "quantile"]),
    "r_alpha": D.FloatDistribution(1e-3, 0.1, log=True),
}

i = 0
for seed, data in make_mc(seeds=range(20), n_train=1500).items():
    simulation_time = time.perf_counter()
    print(i + 1)
    x_train, y_train, x_test, y_test = data

    forest, forest_study = tune_forest(
        x_train, y_train, 3, 100, tree_params, forest_params
    )
    spline, spline_study = tune_spline(
        x_train, y_train, 3, 30, spline_params
    )

    # d = x_train.shape[1]
    # n_knots = 3
    #
    # knots = np.tile(
    #     np.linspace(0, 1, n_knots)[:, None],
    #     (1, d),
    # )
    #
    # spline = TensorSplineRegressor(
    #     degree=3,
    #     knots=knots,
    #     r_alpha=1e-3,
    # ).fit(x_train, y_train)

    #forest.fit(x_train, y_train)

    f_hat_20 = forest.predict(x_test, alpha=(2, 0))
    f_hat_11 = forest.predict(x_test, alpha=(1, 1))
    f_hat_02 = forest.predict(x_test, alpha=(0, 2))

    spline_hat_20 = spline.predict(x_test, alpha=(2, 0))
    spline_hat_11 = spline.predict(x_test, alpha=(1, 1))
    spline_hat_02 = spline.predict(x_test, alpha=(0, 2))

    hessian_test = hessian_f(x_test)

    d20_test = hessian_test[:, 0, 0]
    d11_test = hessian_test[:, 0, 1]
    d02_test = hessian_test[:, 1, 1]

    mse_20[0].append(((f_hat_20 - d20_test) ** 2).mean())
    mse_11[0].append(((f_hat_11 - d11_test) ** 2).mean())
    mse_02[0].append(((f_hat_02 - d02_test) ** 2).mean())
    mae_20[0].append(np.abs(f_hat_20 - d20_test).mean())
    mae_11[0].append(np.abs(f_hat_11 - d11_test).mean())
    mae_02[0].append(np.abs(f_hat_02 - d02_test).mean())

    mse_20[1].append(((spline_hat_20 - d20_test) ** 2).mean())
    mse_11[1].append(((spline_hat_11 - d11_test) ** 2).mean())
    mse_02[1].append(((spline_hat_02 - d02_test) ** 2).mean())
    mae_20[1].append(np.abs(spline_hat_20 - d20_test).mean())
    mae_11[1].append(np.abs(spline_hat_11 - d11_test).mean())
    mae_02[1].append(np.abs(spline_hat_02 - d02_test).mean())

    print('MSE forest:', np.mean(mse_20[0]), 'MAE forest:', np.mean(mae_20[0]))
    print('MSE spline:', np.mean(mse_20[1]), 'MAE spline:', np.mean(mae_20[1]))
    print(time.perf_counter() - simulation_time)

    i += 1

[I 2026-11-02 22:36:45,698] A new study created in memory with name: no-name-68c37b7a-eb8b-4e58-a3c7-908bdabd13e0


1


[I 2026-11-02 22:44:15,780] A new study created in memory with name: no-name-1bbd84df-147b-47a9-8b58-956f5895c55b
[I 2026-11-02 22:44:24,575] A new study created in memory with name: no-name-427ec91a-6d38-4a0b-a91d-bd3614f0279f


MSE forest: 4044.9410301254106 MAE forest: 44.6848991979412
MSE spline: 2028.0569030632648 MAE spline: 32.0325470786112
458.8769422000041
2


[I 2026-11-02 22:49:22,603] A new study created in memory with name: no-name-a1b1c45b-2f3a-4ed5-a0f5-10a5b51e5c1d
[I 2026-11-02 22:49:31,392] A new study created in memory with name: no-name-b06026d9-9a3e-4ffb-96d6-f91a0588753f


MSE forest: 3347.311626966465 MAE forest: 40.09119344874404
MSE spline: 2172.1867846532086 MAE spline: 32.92735556664264
306.81760429998394
3


... (5 outputs left out)

MSE forest: 3087.913944788492 MAE forest: 38.503501240531136
MSE spline: 2225.6475472945012 MAE spline: 33.22081109728532
490.9821397000924
6


[I 2026-11-02 23:18:52,554] A new study created in memory with name: no-name-f58a2eee-d4eb-41bc-beb1-acb51f8a593a
[I 2026-11-02 23:19:02,113] A new study created in memory with name: no-name-cde4b4f1-7b74-4e59-8f9a-ce70a164c955


MSE forest: 3012.6424280121664 MAE forest: 37.82380323406847
MSE spline: 2193.3329734670892 MAE spline: 32.86854679366769
385.945225100033
7


KeyboardInterrupt: 

In [ ]:
np.mean(mse_11[1])